In [1]:
import os

In [2]:
if not os.path.exists("scripts/process_asr_text_tokenizer.py"):
    !wget -P ../scripts/ https://raw.githubusercontent.com/NVIDIA/NeMo/main/scripts/tokenizers/process_asr_text_tokenizer.py

VOCAB_SIZE = 512
TOKENIZER_TYPE = "spe"
SPE_TYPE = "unigram"

!rm -r ../tokenizers/

if not os.path.exists("../tokenizers"):
    os.makedirs("../tokenizers")

!python ../scripts/process_asr_text_tokenizer.py \
    --data_file="../data/transcript_manifest.txt" \
    --data_root="../tokenizers" \
    --tokenizer=$TOKENIZER_TYPE \
    --spe_type=$SPE_TYPE \
    --spe_character_coverage=1.0 \
    --log \
    --vocab_size=$VOCAB_SIZE


--2026-11-24 10:03:45--  https://raw.githubusercontent.com/NVIDIA/NeMo/main/scripts/tokenizers/process_asr_text_tokenizer.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17221 (17K) [text/plain]
Saving to: ‘../scripts/process_asr_text_tokenizer.py.1’

process_asr_text_to 100%[===================>]  16.82K  --.-KB/s    in 0.1s    

2026-11-24 10:03:47 (149 KB/s) - ‘../scripts/process_asr_text_tokenizer.py.1’ saved [17221/17221]

[NeMo I 2026-11-24 10:04:52 sentencepiece_tokenizer:641] Processing ../data/transcript_manifest.txt and store at ../tokenizers/tokenizer_spe_unigram_v512
I0000 00:00:1795511092.980660    3181 sentencepiece_trainer.cc:166] Running command: --input=../data/transcript_manifest.txt --model_prefix=../tokenizers/tokenizer_spe_unigram_v51

In [9]:
import re
import unicodedata

TONE_MARKS = {
    "1": "\u0301",  # high: acute
    "2": "\u0300",  # low: grave
    "3": "\u030C",  # rising: caron
    "4": "\u0302",  # falling: circumflex
    "5": "\u0304",  # mid: macron
}
VOWELS = set("aeiouɛɔə")


def restore_tones(text: str) -> str:
    def restore_token(match: re.Match) -> str:
        restored = []
        syllable_start = 0

        for char in unicodedata.normalize("NFD", match.group()):
            mark = TONE_MARKS.get(char)
            if mark is None or not restored:
                restored.append(char)
                continue

            # Put the tone on the most recent vowel in this syllable.
            # For syllabic consonants, use the most recent base character.
            target = next(
                (
                    i for i in range(len(restored) - 1, syllable_start - 1, -1)
                    if restored[i].lower() in VOWELS
                ),
                next(
                    (
                        i for i in range(len(restored) - 1, syllable_start - 1, -1)
                        if not unicodedata.category(restored[i]).startswith("M")
                    ),
                    None,
                ),
            )

            if target is None:
                restored.append(char)
            else:
                restored.insert(target + 1, mark)
                syllable_start = len(restored)

        return unicodedata.normalize("NFC", "".join(restored))

    return re.sub(r"\S+", restore_token, text)

In [11]:
from nemo.collections.common.tokenizers import SentencePieceTokenizer
from fongbe.utils.text_tools import syllabify

tok_path = "../tokenizers/tokenizer_spe_unigram_v512/tokenizer.model"
tokenizer = SentencePieceTokenizer(model_path=tok_path)

samples = [
    "a hɛn ɔ1 a na3 gbeje3 akɔn1ta e1lɔ xa1 mi2 a",
    "mi3 mɛ bi3 wɛ jɔhɔn ɖo sinsin wɛ zan3zan3 din bɔ mi3 jlo1 bo1 na1 lɛ wu3 a3",
    "un mɔ axi ɖaxo1 ɖokpo1 bɔ e1 jɛ tawun bɔ mɛ lɛ1ɛ ɖo te ɖo axi ɔ1 mɛ",
]

for raw in samples:
    
    syl = syllabify(restore_tones(raw))
    ids = tokenizer.text_to_ids(raw)
    pieces = tokenizer.ids_to_tokens(ids)
    decoded = tokenizer.ids_to_text(ids)

    print(f"Raw     : {raw}")
    print(f"Syllable: {syl}")
    print(f"Pieces  : {pieces}")
    print(f"Decoded : {decoded}")
    assert tokenizer.unk_id not in ids, f"UNK token found in: {syl}"
    # assert decoded == syl, f"Round-trip mismatch: {decoded!r} != {syl!r}"
    print("-" * 50)

print(f"Vocab size: {tokenizer.vocab_size} | All round-trip checks passed.")

Raw     : a hɛn ɔ1 a na3 gbeje3 akɔn1ta e1lɔ xa1 mi2 a
Syllable: a hɛn ɔ1 a na3 gbe je3 a kɔn1 ta e1 lɔ xa1 mi2 a
Pieces  : ['▁a', '▁hɛn', '▁ɔ', '1', '▁a', '▁na', '3', '▁gbe', 'j', 'e', '3', '▁a', 'k', 'ɔn', '1', 't', 'a', '▁e', '1', 'l', 'ɔ', '▁xa', '1', '▁mi', '2', '▁a']
Decoded : a hɛn ɔ1 a na3 gbeje3 akɔn1ta e1lɔ xa1 mi2 a
--------------------------------------------------
Raw     : mi3 mɛ bi3 wɛ jɔhɔn ɖo sinsin wɛ zan3zan3 din bɔ mi3 jlo1 bo1 na1 lɛ wu3 a3
Syllable: mi3 mɛ bi3 wɛ jɔ hɔn ɖo sin sin wɛ zan3 zan3 din bɔ mi3 jlo1 bo1 na1 lɛ wu3 a3
Pieces  : ['▁mi', '3', '▁mɛ', '▁bi', '3', '▁wɛ', '▁jɔ', 'h', 'ɔn', '▁ɖo', '▁sin', 's', 'i', 'n', '▁wɛ', '▁zan', '3', 'z', 'a', 'n', '3', '▁din', '▁bɔ', '▁mi', '3', '▁jlo', '1', '▁bo', '1', '▁na', '1', '▁lɛ', '▁wu', '3', '▁a', '3']
Decoded : mi3 mɛ bi3 wɛ jɔhɔn ɖo sinsin wɛ zan3zan3 din bɔ mi3 jlo1 bo1 na1 lɛ wu3 a3
--------------------------------------------------
Raw     : un mɔ axi ɖaxo1 ɖokpo1 bɔ e1 jɛ tawun bɔ mɛ lɛ1ɛ ɖo te ɖo axi ɔ1 mɛ